# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/colab/thesis/hqs
CWD = Path.cwd()

In [ ]:
# @title 🧩 Package Installation
%%capture
!pip install uv
!uv pip install scispacy
# !uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
# !uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
!uv pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz
# !uv pip install evaluation/en_core_sci_scibert-0.5.4.tar.gz

# 🔬 Evaluation

---

⚠️ [ _Press_ `Ctrl+M .` _before proceeding!_ ]

---

In [ ]:
import spacy
import scispacy
import numpy as np
import pandas as pd
from pathlib import Path
from scispacy.abbreviation import AbbreviationDetector
from scispacy.linking import EntityLinker

%cd /content/drive/MyDrive/colab/thesis/hqs
CWD = Path.cwd()

In [ ]:
# @title 📊 Test Dataset
test_df = pd.read_json("data/test-gen3-qwen.jsonl", lines=True)
test_df.info()

In [ ]:
nlp = spacy.load("en_core_sci_lg", enable=["ner"])
# nlp = spacy.load("en_core_sci_scibert")
# nlp.add_pipe("abbreviation_detector")
# nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})

In [ ]:
# @title Computes Faithfulness-adjusted Recall (FaR) and Concept F1 (CF1)
def _extract_entities(text, nlp_model):
    doc = nlp_model(text)
    return set([
        word
        for ent in doc.ents
        for word in str(ent.text).strip().lower().split()
    ])


def _compute_metrics(source, reference, generated, nlp_model):
    # Step 1: Extract entities as sets for each document
    S = _extract_entities(source, nlp_model)
    R = _extract_entities(reference, nlp_model)
    M = _extract_entities(generated, nlp_model)

    # Step 2: Map to Venn diagram intersections
    # Region C: Entities shared by Source, Reference, and Generated
    C = S.intersection(R).intersection(M)

    # Regions B + C: Entities shared by Source and Reference
    B_plus_C = S.intersection(R)

    # Regions C + F: Entities shared by Reference and Generated
    C_plus_F = R.intersection(M)

    # Step 3: Compute Faithfulness-adjusted Recall (FaR)
    # FaR = C / (B + C)
    if len(B_plus_C) > 0:
        far_score = len(C) / len(B_plus_C)
    else:
        far_score = 0.0  # Handle division by zero

    # Step 4: Compute Concept Recall and Concept Precision
    # Concept Rec = (C + F) / (B + C + E + F)
    # Note: R is equivalent to regions B + C + E + F
    if len(R) > 0:
        concept_rec = len(C_plus_F) / len(R)
    else:
        concept_rec = 0.0

    # Concept Prec = (C + F) / (C + D + F + G)
    # Note: M is equivalent to regions C + D + F + G
    if len(M) > 0:
        concept_prec = len(C_plus_F) / len(M)
    else:
        concept_prec = 0.0

    # Step 5: Compute Concept F1
    if (concept_rec + concept_prec) > 0:
        concept_f1 = (2 * concept_rec * concept_prec) / (concept_rec + concept_prec)
    else:
        concept_f1 = 0.0

    return far_score, concept_f1

In [ ]:
_extract_entities("23 surgeries and counting......lower lip birthmark, have tried all options out the there and guess what still have it, continues to grow back.....any suggestions? Is there a cure coming in the next few years hopefully?", nlp)

In [ ]:
def compute_metrics(sources, refs, preds, nlp_model):
    far_scores = []
    cf1_scores = []

    for source, ref, pred in zip(sources, refs, preds):
        far, cf1 = _compute_metrics(source, ref, pred, nlp_model)
        far_scores.append(far)
        cf1_scores.append(cf1)

    return np.mean(far_scores), np.mean(cf1_scores)

In [ ]:
far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["biobart_gen"].to_list(),
    nlp_model=nlp
)

print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

In [ ]:
far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["qwen_gen"].to_list(),
    nlp_model=nlp
)

print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

In [ ]:
far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["qwen3x_gen"].to_list(),
    nlp_model=nlp
)

print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

In [ ]:
# @title Pegasus
far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["pegasus_gen"].to_list(),
    nlp_model=nlp
)
print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["qwen3_gen"].to_list(),
    nlp_model=nlp
)
print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["qwen3x_gen"].to_list(),
    nlp_model=nlp
)
print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")

In [ ]:
far, cf1 = compute_metrics(
    test_df["source"].to_list(),
    test_df["summary"].to_list(),
    test_df["pegasus_gen"].to_list(),
    nlp_model=nlp
)
print(f"FaR Score: {far:.4f}")
print(f"Concept F1 Score: {cf1:.4f}")